
# Embodyment aware collaborated learning

In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys

module_path = os.path.abspath(os.path.join("./src"))  # or the path to your source code
sys.path.insert(0, module_path)

# Setting up the learning environment

For training the agents, the
[gymnasium](https://github.com/Farama-Foundation/Gymnasium) packagage is
utilized. The gymnasium package is a collection of environments that can be
easily set up and used for training agents. 


In [2]:
import gymnasium
from itertools import count
import torch
from collections import namedtuple
import matplotlib.pyplot as plt
from federated_learner import device
from federated_learner.agent import AgentConfig, DeepQNetwork, DQNAgent
from federated_learner.visualization import plot_durations, plot_reward
from federated_learner.utils import test_agent, fill_buffer

SEED = 42
torch.manual_seed(SEED)

In [3]:
# The following device is used for training the agents:
print(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

mps


# Train on simple DeepQAgent

## Initialize the environment

Initialize the environment without rendering and fetch the dimensiont of the state and action
space.

In [4]:
env = gymnasium.make("CartPole-v1")
state, info = env.reset()
state_dim = len(state)
action_dim = env.action_space.n
print(f"State dimension: {state_dim} and Action dimension: {action_dim}")

State dimension: 4 and Action dimension: 2


In [5]:
agent_config = AgentConfig(
    state_dim=state_dim,
    action_dim=action_dim,
    learning_rate=1e-4,
    tau=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_decay=1000,
    epsilon_end=0.05,
    buffer_size=1000,
    batch_size=64,
)

The utilized optimizer is the
[AdamW](https://pytorch.org/docs/stable/optim.html#torch.optim.AdamW) optimizer.

It uses a simple neural network with two hidden layers of 8 units each.

In [6]:
agent = DQNAgent(agent_config, DeepQNetwork)

In [7]:
num_episodes = 1500

AverageReward = namedtuple("AverageReward", ("episode", "reward"))
average_rewards = []
std_deviation_rewards = []
episode_durations = []

print("Filling the buffer with random actions")
fill_buffer(env, agent, SEED)
print("Buffer filled")
for i_episode in range(num_episodes):
    # Initialize the environment and get its state
    if i_episode % 25 == 0:
        average_reward, std_deviation_reward = test_agent(env, agent, SEED)
        print(
            f"Episode {i_episode} --> Average Total Reward (Evaluation): {average_reward}"
        )
        average_rewards.append(AverageReward(i_episode, average_reward))
        std_deviation_rewards.append(AverageReward(i_episode, average_reward))
    # Initialize the environment and get its state
    state, info = env.reset(seed=SEED)
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    for t in count():
        action = agent.select_action(state)
        observation, reward, terminated, truncated, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device, dtype=torch.float32)
        done = terminated or truncated

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(
                observation, dtype=torch.float32, device=device
            ).unsqueeze(0)

        # Store the transition in memory
        agent.remember(state, action, next_state, reward)

        # Move to the next state
        state = next_state

        # Perform one step of the optimization (on the policy network)
        agent.optimize_model()
        agent.soft_update()

        if done:
            episode_durations.append(t + 1)
            break


print("Complete")

Filling the buffer with random actions


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 1449.88it/s]


Buffer filled
Episode 0 --> Average Total Reward (Evaluation): 13.77
Episode 25 --> Average Total Reward (Evaluation): 9.35
Episode 50 --> Average Total Reward (Evaluation): 8.54
Episode 75 --> Average Total Reward (Evaluation): 8.41
Episode 100 --> Average Total Reward (Evaluation): 8.22
Episode 125 --> Average Total Reward (Evaluation): 10.28
Episode 150 --> Average Total Reward (Evaluation): 10.3
Episode 175 --> Average Total Reward (Evaluation): 8.28
Episode 200 --> Average Total Reward (Evaluation): 10.36
Episode 225 --> Average Total Reward (Evaluation): 10.33
Episode 250 --> Average Total Reward (Evaluation): 10.4
Episode 275 --> Average Total Reward (Evaluation): 14.96
Episode 300 --> Average Total Reward (Evaluation): 11.11
Episode 325 --> Average Total Reward (Evaluation): 11.21
Episode 350 --> Average Total Reward (Evaluation): 11.07
Episode 375 --> Average Total Reward (Evaluation): 11.1
Episode 400 --> Average Total Reward (Evaluation): 11.18
Episode 425 --> Average Total 

KeyboardInterrupt: 

In [ ]:
plot_durations(episode_durations, show_result=True)
plt.show()

In [ ]:
plot_reward(average_rewards, std_deviation_rewards)

### Visualize the best learned policy

In [ ]:
# Initialise the environment
# env = gymnasium.make("CartPole-v1")
#
# # Initialize the environment and get its state
# state, info = env.reset()
# state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
# for t in count():
#     action = agent.select_action(state)
#     observation, reward, terminated, truncated, _ = env.step(action.item())
#     reward = torch.tensor([reward], device=device)
#     done = terminated or truncated
#
#     if terminated:
#         next_state = None
#     else:
#         next_state = torch.tensor(
#             observation, dtype=torch.float32, device=device
#         ).unsqueeze(0)
#
#     # Move to the next state
#     state = next_state
#
# env.close()